In [7]:
import os
import matplotlib.pyplot as plt
from cellpose import models
import import_images  # seu script de carregar imagens

# Força uso de CPU
models.use_gpu(False)

# Caminho da pasta com imagens
base_dir = "/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/"

# Encontra todas as imagens .tif
image_paths = import_images.encontrar_imagens_tiff(base_dir)

# Inicializa modelo Cellpose-SAM
model = models.Cellpose(model_type='cyto')  # grayscale

# Parâmetros para imagens grayscale
channels = [0, 0]  # [cytoplasma, nuclear], 0=grayscale, 1=fluorescence

# Pasta de saída
output_dir = os.path.join(os.getcwd(), "resultados", "cellpose_sam")
os.makedirs(output_dir, exist_ok=True)

# Loop em todas as imagens
for idx, image_path in image_paths.items():
    # Carrega imagem
    image = import_images.carregar_imagem_por_indice(image_paths, idx)
    if image is None:
        continue

    # Avaliação com Cellpose
    masks, flows, styles, diams = model.eval(
        image,
        diameter=None,  # None = automático
        channels=channels,
        do_3D=False
    )

    # Gera overlay
    overlay = model.make_overlay(image, masks, flows[0], channels=channels)

    # Nome base da imagem
    nome_original = os.path.splitext(os.path.basename(image_path))[0]

    # ---------- Salva overlay ----------
    overlay_path = os.path.join(output_dir, f"{nome_original}_overlay.png")
    plt.imsave(overlay_path, overlay)
    print(f"Overlay salvo em: {overlay_path}")

    # ---------- Salva subplot (input + overlay) ----------
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(image, cmap="gray")
    ax[0].axis("off")
    ax[0].set_title("Input image")

    ax[1].imshow(overlay)
    ax[1].axis("off")
    ax[1].set_title("Prediction + overlay")

    subplot_path = os.path.join(output_dir, f"{nome_original}_subplot.png")
    plt.savefig(subplot_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Subplot salvo em: {subplot_path}")


AttributeError: module 'cellpose.models' has no attribute 'use_gpu'